# 🐔 Agricare AI Engine (Gemini 2.5 Flash + RAG)

This notebook implements the complete **Agricare AI Conversational Diagnostic Engine**. It uses Google's **Gemini 2.5 Flash API** combined with a **Retrieval-Augmented Generation (RAG)** architecture and a local veterinary knowledge base of 15 poultry diseases.

### ✨ Key Highlights
- **Safety-First Triage:** Strictly asks 1-3 clarifying questions before diagnosing.
- **Multilingual Support:** Natively detects and communicates in **English, Hausa, Yoruba, Igbo, and Pidgin**.
- **Structured Output:** Forces strict JSON output for predictability.
- **Offline Fallback:** Heuristic keyword matching guarantees 100% uptime when offline.

## 1. Setup & Dependencies
Run this cell to install `requests` (used for the Gemini REST API).

In [ ]:
!pip install requests

## 2. Define the Agricare AI Engine Class
This class handles API inference, knowledge base retrieval, triage prompting, and local offline fallbacks.

In [ ]:
import os
import json
import re
import requests

class AgriCareEngine:
    """Conversational AI RAG diagnostic engine for poultry health using Gemini 2.5 Flash with local fallback."""

    def __init__(self, api_key: str = None):
        print("🧠 Initializing Conversational RAG Agricare AI engine...")
        self.api_key = api_key or os.getenv("GEMINI_API_KEY")
        if not self.api_key:
            print("⚠️ WARNING: GEMINI_API_KEY is not set. Using local RAG fallback engine.")

        # Load local knowledge base
        self.kb_path = "data/knowledge_base.json"
        if os.path.exists(self.kb_path):
            with open(self.kb_path, "r", encoding="utf-8") as f:
                self.knowledge = json.load(f)
            print(f"✅ Loaded {len(self.knowledge)} diseases from {self.kb_path}")
        else:
            self.knowledge = []
            print("⚠️ WARNING: data/knowledge_base.json not found!")

    def fallback_process(self, text: str) -> dict:
        """Fallback symptom matching when Gemini is offline or unavailable."""
        text_lower = text.lower()
        
        # Language detection heuristic
        lang = "en"
        lang_indicators = {
            "ha": ["kaji", "zawo", "mutu", "hanci", "tari", "kore", "fari", "baki", "ciki", "da"],
            "yo": ["adìẹ", "arun", "gbẹ́", "ẹjẹ", "omi", "ewé", "orí", "enà", "ní", "tó"],
            "ig": ["ọkụkọ", "ọrịa", "nsị", "ọbara", "mmiri", "nri", "isi", "taa", "na", "ndụ"],
            "pcm": ["dey", "wey", "go", "fit", "chop", "sabi", "abeg", "shit", "blood", "body", "dem"]
        }
        words = set(re.findall(r'\b\w+\b', text_lower))
        for l, indicators in lang_indicators.items():
            if any(ind in words for ind in indicators):
                lang = l
                break
                
        # Keyword matching
        best_match = None
        max_matches = 0
        for disease in self.knowledge:
            matches = 0
            symptom_list = disease.get("symptoms", {}).get(lang, []) + disease.get("symptoms", {}).get("en", [])
            for symptom in symptom_list:
                if symptom.lower() in text_lower:
                    matches += 2
                elif len(set(text_lower.split()).intersection(set(symptom.lower().split()))) >= 2:
                    matches += 1
            for ew in disease.get("escalation_words", []):
                if ew.lower() in text_lower:
                    matches += 3
            if matches > max_matches:
                max_matches = matches
                best_match = disease
                
        if best_match:
            name = best_match["names"].get(lang, best_match["names"]["en"])
            advice = best_match["advice"].get(lang, best_match["advice"]["en"])
            severity = best_match.get("severity", "MEDIUM")
            urgency = "RED" if severity == "CRITICAL" else ("ORANGE" if severity == "HIGH" else "GREEN")
            escalate = severity == "CRITICAL"
            answer = f"[{urgency}] {name}\n\nAdvice: {advice}\n\n*Note: Running in offline backup mode.*"
            return {"language": lang, "disease_id": best_match["id"], "disease_name": name, "urgency": urgency, "escalate": escalate, "answer": answer}
            
        return {"language": lang, "disease_id": None, "disease_name": None, "urgency": "GREEN", "escalate": False, "answer": "Please isolate sick birds, provide fresh water/warmth, and contact a vet."}

    def process(self, text: str, history: list = None) -> dict:
        """Processes user input using Gemini 2.5 Flash with strict triage constraints."""
        if not self.api_key:
            return self.fallback_process(text)
            
        kb_str = json.dumps(self.knowledge, ensure_ascii=False, indent=2)
        url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={self.api_key}"
        
        system_prompt = (
            "You are Agricare AI, an expert poultry veterinarian for Nigerian farmers.\n"
            f"Knowledge base containing 15 diseases:\n{kb_str}\n\n"
            "Instructions:\n"
            "1. Detect language (en, ha, yo, ig, pcm).\n"
            "2. Act as a veterinary triage assistant. When a farmer FIRST describes a vague problem, DO NOT immediately diagnose. MUST ask 1-2 clarifying questions (age, duration, diarrhea, vaccines).\n"
            "3. When asking questions, set `disease_id` and `disease_name` to null, and `urgency` to 'GREEN'.\n"
            "4. Only after answering questions / enough context, match symptoms to the disease and provide final diagnosis/advice in the farmer's language.\n"
            "5. Output strict JSON: {'language': '..', 'disease_id': '..', 'disease_name': '..', 'urgency': 'RED|ORANGE|YELLOW|GREEN', 'escalate': true|false, 'answer': '..'}\n"
            "Always output raw JSON ONLY."
        )
        
        contents = []
        if history:
            for msg in history:
                contents.append({"role": msg.get("role", "user"), "parts": [{"text": msg.get("text", "")}]})
        contents.append({"role": "user", "parts": [{"text": text}]})
        
        payload = {"system_instruction": {"parts": [{"text": system_prompt}]}, "contents": contents, "generationConfig": {"temperature": 0.2}}
        
        try:
            resp = requests.post(url, headers={"Content-Type": "application/json"}, json=payload, timeout=15)
            if resp.status_code == 200:
                raw = resp.json()["candidates"][0]["content"]["parts"][0]["text"].strip()
                if raw.startswith("```"): raw = re.sub(r"^```[a-z]*\n|```$", "", raw, flags=re.MULTILINE).strip()
                return json.loads(raw)
        except Exception as e:
            print(f"⚠️ Gemini Error: {e}")
            
        return self.fallback_process(text)

## 3. Run Interactive Diagnostic Session
Enter your Google Gemini API Key below (or leave blank to test the offline fallback engine), then run the interactive test loop.

In [ ]:
# Paste your API key inside the quotes
API_KEY = ""

engine = AgriCareEngine(api_key=API_KEY)

print("\n👋 Welcome to the Agricare AI Diagnostic Shell!")
print("Type 'quit' to exit.\n")

history = []
while True:
    user_input = input("👨‍🌾 Farmer: ")
    if user_input.lower() in ["quit", "exit"]:
        break
        
    result = engine.process(user_input, history=history)
    print(f"\n🤖 Agricare AI [{result.get('urgency')}]: {result.get('answer')}\n")
    
    history.append({"role": "user", "text": user_input})
    history.append({"role": "model", "text": result.get('answer')})